# 01 — Data Ingestion

**Phase 1 of the Weather → Climate → Wildfire pipeline.**

This notebook is the **single entry point** for loading every raw source the
project depends on, validating it, and writing a clean, analysis-ready copy
to `data/interim/`. All heavy lifting lives in `src/weather/ingestion.py`;
this notebook orchestrates, inspects and reports.

---

## Data sources

| # | Source | File(s) | Purpose |
|---|---|---|---|
| 1 | **Open-Meteo archive** | `data/raw/openmeteo/hourly_weather.csv`, `daily_weather.csv` | Weather history (core) |
| 2 | **WorldPop population** | `data/raw/population/population_YYYY.csv` | Human-activity proxy |
| 3 | **OSM road network** | `data/raw/osm_roads/static_city_infrastructure.csv` | Human-access proxy |
| 4 | **MODIS NDVI (GEE)** | `data/raw/earth_engine/aze_ndvi_2020_2026.csv` | Vegetation dryness |
| 5 | **NASA FIRMS VIIRS** | `data/raw/nasa_firms/viirs-snpp_*.csv` | Historical wildfire labels |
| 6 | **Lightning climatology** | `data/raw/lightning/azerbaijan_lightning_*.csv` | Natural ignition source |
| 7 | **Forest boundaries** | `data/raw/forest_boundaries/azerbaijan.kmz` | Fuel map |
| 8 | **ESA WorldCover** | `data/raw/earth_engine/aze_landcover.tif` | Landcover per city |

## Outputs

Every loader writes to `data/interim/`:

```
weather_hourly_raw.csv      # Core weather — 5 cities × hourly grid
weather_daily_raw.csv       # Daily rain sum
cities_reference.csv        # Name, lat, lon for 16 study cities
population.csv              # Stacked 2020–2026
roads.csv                   # Human-access road density
ndvi.csv                    # Daily NDVI per city
firms.csv                   # Fire hotspots + combined timestamp
lightning.csv               # Grid-month thunder-hours
forest_boundaries.geojson   # (optional) geopandas / fiona required
landcover_at_cities.csv     # (optional) rasterio required
```

## 1. Setup

Make `src/` importable from the notebook and pull in the ingestion helpers.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the project root importable no matter where Jupyter was launched from
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

from src.weather import ingestion
from src.utils.config import (
    AZERBAIJAN_CITIES, INTERIM_DIR, RAW_OPENMETEO,
    HOURLY_WEATHER_VARS, FORECAST_TARGETS,
)
from src.utils.logging_utils import get_logger
logger = get_logger("nb.01_ingestion")
logger.info("Setup complete - %d cities in scope", len(AZERBAIJAN_CITIES))

## 2. Weather — Open-Meteo historical archive

The Open-Meteo REST API serves hourly history from 2020-01-01 to the present
for every lat/lon, free of charge. We cached a 2020–2026 snapshot under
`data/raw/openmeteo/` so the notebook runs offline; the same function can
refresh directly from the API when network access is available.

### 2a. Load from disk (default path)

In [ ]:
weather_hourly, weather_daily = ingestion.load_weather_from_disk()
weather_hourly.head(3)

### 2b. Validate structural integrity

`validate_weather_df` reports row counts, city coverage, date range, missing-cell
percentage, and — crucially — whether the hourly / daily cadence has any gaps.

In [ ]:
hourly_report = ingestion.validate_weather_df(weather_hourly, kind="hourly")
daily_report  = ingestion.validate_weather_df(weather_daily,  kind="daily")

summary = pd.DataFrame({
    "hourly": {k: v for k, v in hourly_report.items() if k != "timestep_gaps_per_city"},
    "daily":  {k: v for k, v in daily_report.items()  if k != "timestep_gaps_per_city"},
})
summary

In [ ]:
# Gap analysis per city (0 = perfect cadence)
pd.DataFrame({
    "hourly_gaps": hourly_report["timestep_gaps_per_city"],
    "daily_gaps":  daily_report["timestep_gaps_per_city"],
}).fillna(0).astype(int)

### 2c. (Optional) Refresh directly from the Open-Meteo API

Uncomment the cell below to re-download. Requires `openmeteo_requests`,
`requests_cache` and `retry_requests`, plus network access.

In [ ]:
# hourly_api, daily_api = ingestion.fetch_weather_from_api()
# hourly_api.head()

## 3. Cities reference table

A tiny but important artefact: the canonical mapping from city name to
centroid coordinates, used by every downstream join.

In [ ]:
from src.utils.config import cities_as_records
cities_df = pd.DataFrame(cities_as_records())
cities_df

## 4. Auxiliary sources

Each loader is independent and validated in isolation — a failure in one
source never takes down the others.

### 4a. Population (WorldPop, rasterised to city centroids)

In [ ]:
population = ingestion.load_population()
population.pivot_table(index="City", columns="Year", values="Pop_Density").round(1)

### 4b. Roads (OSM human-access within 5 km of each city)

In [ ]:
roads = ingestion.load_roads_static()
roads.sort_values("human_access_road_meters", ascending=False)

### 4c. NDVI (MODIS, per city, Earth Engine export)

In [ ]:
ndvi = ingestion.load_ndvi()
print(ndvi.groupby("City")["NDVI"].agg(["min", "mean", "max"]).round(3))
ndvi.head(3)

### 4d. NASA FIRMS (VIIRS wildfire hotspots)

In [ ]:
firms = ingestion.load_firms()
print("Annual fire counts:")
print(firms["acq_date"].dt.year.value_counts().sort_index())
firms[["latitude", "longitude", "acq_timestamp", "bright_ti4", "frp", "confidence", "daynight"]].head(3)

### 4e. Lightning (climatology grid)

In [ ]:
lightning = ingestion.load_lightning()
print("Thunder-hours per month (all years pooled):")
print(lightning.groupby("month")["thunder_hours"].sum())
lightning.head(3)

### 4f. Forest boundaries and landcover (require `geopandas` / `rasterio`)

These cells skip themselves with a clear message if the optional geo libraries
are missing — the pipeline still produces complete tabular outputs.

In [ ]:
try:
    forests = ingestion.load_forest_boundaries()
    print(f"Forests: {len(forests)} polygons, CRS={forests.crs}")
    forests.head(3)
except RuntimeError as exc:
    print(f"[skipped] {exc}")
    forests = None
forests

In [ ]:
try:
    landcover = ingestion.sample_landcover_at_cities()
    display(landcover)
except RuntimeError as exc:
    print(f"[skipped] {exc}")
    landcover = None

## 5. Run the full pipeline & persist interim outputs

The orchestrator calls every loader, catches per-source failures and writes
clean copies to `data/interim/`. Use `fmt='parquet'` in production (much
smaller, typed); CSV is the portable default.

In [ ]:
outputs = ingestion.run_ingestion_pipeline(use_api=False, fmt="csv", include_geo=True)

summary_rows = []
for name, df in outputs.items():
    summary_rows.append({"dataset": name, "rows": len(df), "cols": df.shape[1] if hasattr(df, 'shape') else None})
pd.DataFrame(summary_rows)

In [ ]:
# Inventory of interim/ — everything downstream Phase 2 will consume
from pathlib import Path
interim = sorted(INTERIM_DIR.glob("*"))
pd.DataFrame({
    "file": [p.name for p in interim],
    "size_MB": [round(p.stat().st_size / 1024 / 1024, 3) for p in interim],
})

## 6. Quick visual QC

A first sanity check that the hourly weather pulled from Open-Meteo looks
physically reasonable — temperature should show a clean annual cycle, and
the cities should all share the same envelope with local shifts.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 4))
for city, g in weather_hourly.groupby("City"):
    # Daily mean for legibility across ~55k hours
    daily = g.set_index("date")["temperature_2m"].resample("D").mean()
    ax.plot(daily.index, daily.values, label=city, lw=0.8, alpha=0.75)

ax.set_title("Daily mean 2 m temperature, 2020–2026")
ax.set_ylabel("°C"); ax.set_xlabel("date")
ax.legend(ncol=5, fontsize=8, loc="lower center")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Missingness map per variable and per city — shows if any source station dropped a column
miss = (
    weather_hourly
    .drop(columns=["date"])
    .groupby("City")
    .apply(lambda g: g.isna().mean().round(3), include_groups=False)
)
miss.style.background_gradient(cmap="Reds", vmin=0, vmax=0.2) if hasattr(miss, "style") else miss

## 7. Hand-off to Phase 2

Everything Phase 2 (weather cleaning + modelling) needs is now in
`data/interim/`:

1. **`weather_hourly_raw.csv`** — the core forecasting dataset
2. **`weather_daily_raw.csv`** — daily rain-sum reference
3. **`cities_reference.csv`** — 16 study locations
4. **Auxiliary frames** — population, roads, ndvi, firms, lightning
   (the wildfire system in Phase 4 will consume these)

Forecast targets (per `src.utils.config.FORECAST_TARGETS`):

In [ ]:
print("Forecast targets:", FORECAST_TARGETS)
print("\nTargets available in weather_hourly:",
      [t for t in FORECAST_TARGETS if t in weather_hourly.columns])
missing = [t for t in FORECAST_TARGETS if t not in weather_hourly.columns]
print("Missing:", missing or "(none)")

---
**Phase 1 ✅ complete.** Move to `02_weather_cleaning.ipynb`.